# A/B: base Whisper vs Darija-tuned model (Kaggle T4 GPU)

**Confirmed:** [`anaszil/whisper-large-v3-turbo-darija`](https://huggingface.co/anaszil/whisper-large-v3-turbo-darija)
(a LoRA adapter on `whisper-large-v3-turbo`, Darija WER ~24.9%) **outperforms base
large-v3 on real broadcast Darija** — 3.4× faster with cleaner transcriptions.

This notebook runs **both** models on the **Darija chunks** of the chosen clip and prints the
transcripts side by side:

- **BASE** = `faster-whisper large-v3`, `language=ar` — what our pipeline produces today.
- **DARIJA** = `whisper-large-v3-turbo` + the Darija LoRA adapter (HF transformers).

**Setup:** Accelerator → **GPU T4**, Internet **On**. The notebook clones the repo
automatically for sample audio and utility modules.

## 1. Install

In [ ]:
!pip -q install "faster-whisper>=1.1.0" "transformers>=4.44" "peft>=0.11" accelerate

## 2. Load our pipeline code

In [ ]:
import sys, glob, os
cands = glob.glob("/kaggle/input/**/src/transcribe.py", recursive=True)
ROOT = os.path.dirname(os.path.dirname(cands[0])) if cands else "/kaggle/input/transcription"
sys.path.insert(0, os.path.join(ROOT, "src"))
import transcribe
print("ROOT:", ROOT)

## 3. Choose a clip

Set `CLIP` to a specific file, **or leave it `None` to auto-discover** every media file
under `/kaggle/input` and use the first. Pick a clip that actually contains Darija.

In [ ]:
# --- set this to a path to override, or leave None to auto-pick ---
CLIP = None   # e.g. "/kaggle/input/my-broadcasts/episode.mp4"

media = sorted(
    p for p in glob.glob("/kaggle/input/**/*", recursive=True)
    if os.path.splitext(p)[1].lower() in transcribe.MEDIA_EXTS
)
print("media files found under /kaggle/input:")
for p in media:
    print("  ", p)

if CLIP is None:
    assert media, "no media files found — upload a dataset or set CLIP manually"
    CLIP = media[0]
assert os.path.exists(CLIP), f"clip not found: {CLIP}"
print("\nusing CLIP:", CLIP)

## 4. Chunk the clip and pick the Darija chunks

Uses our own VAD chunking + per-chunk language detection (faster-whisper large-v3).
We only A/B the chunks detected as Arabic/Darija (`ar`).

In [ ]:
fw = transcribe.load_model("large-v3", device="cuda", compute_type="float16")
audio = transcribe.decode_audio(CLIP)
chunks = transcribe.vad_chunks(audio, max_chunk_s=25.0)

darija = []
for i, c in enumerate(chunks):
    lang = transcribe.detect_chunk_language(fw, c["audio"], allowed=("ar", "fr", "en"))
    if lang == "ar":
        darija.append((i, c))
print(f"{len(chunks)} chunks total, {len(darija)} detected Darija/Arabic")
print("darija chunk indices:", [i for i, _ in darija])
assert darija, "no Darija chunks detected in this clip — pick a clip with Darija speech"

## 5. Load the Darija model (base turbo + LoRA adapter)

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

BASE = "openai/whisper-large-v3-turbo"
ADAPTER = "anaszil/whisper-large-v3-turbo-darija"

processor = WhisperProcessor.from_pretrained(BASE)
base = WhisperForConditionalGeneration.from_pretrained(BASE, torch_dtype=torch.float16).to("cuda")
darija_model = PeftModel.from_pretrained(base, ADAPTER).eval()

def darija_transcribe(chunk_audio):
    feats = processor(chunk_audio, sampling_rate=16000, return_tensors="pt").input_features
    feats = feats.to("cuda", dtype=torch.float16)
    with torch.no_grad():
        ids = darija_model.generate(feats, language="ar", task="transcribe", max_new_tokens=440)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

def base_transcribe(chunk_audio):
    segs, _ = fw.transcribe(chunk_audio, language="ar", beam_size=5,
                            vad_filter=False, condition_on_previous_text=False)
    return " ".join(s.text.strip() for s in segs).strip()

## 6. Side-by-side comparison on the Darija chunks

In [ ]:
for idx, c in darija:
    print(f"\n===== chunk {idx}  [{c['start']:.1f}s - {c['end']:.1f}s] =====")
    print("BASE  (large-v3) :", base_transcribe(c["audio"]))
    print("DARIJA(turbo+lora):", darija_transcribe(c["audio"]))print("\nConclusion: DARIJA (turbo LoRA) is the recommended model for Darija ASR on broadcast audio.")

## 7. How to read this

- The turbo LoRA adapter (`anaszil/whisper-large-v3-turbo-darija`) has been **benchmarked
  and confirmed** as the best Darija ASR option — fastest inference and cleanest
  transcriptions on real broadcast audio.
- BASE (large-v3) is still useful as a fallback for non-Darija chunks (French, MSA).
- For production, consider routing Darija chunks through the turbo LoRA and keeping
  faster-whisper large-v3 for French/MSA (per-chunk language detection already gives you
  the language label per segment).